In [43]:
# 강남구 데이터 10년치 수집함
import pandas as pd
import numpy as np
import requests
from tqdm import tqdm
from dotenv import load_dotenv
import os
import glob

In [44]:
load_dotenv()
api_key = os.getenv("KAKAO_API_KEY")

In [45]:

# 사용할 컬럼만 지정 (NO 컬럼 제외)
columns_to_use = [
    '시군구', '단지명', '전용면적(㎡)', '계약년월', '계약일',
    '거래금액(만원)', '동', '층', '건축년도', '도로명'
]

# 병합할 CSV 파일이 들어있는 폴더 경로
folder_path = "../../data/raw/apt_sale"
gangnam_path = "../../data/raw/apt_sale/gangnam"
# 해당 경로의 모든 .csv 파일 리스트 얻기
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
# 병합할 데이터프레임 저장할 리스트
df_list = []

# 각 CSV 파일을 순회하며 읽기
for file in csv_files:
    try:
        df = pd.read_csv(
            file,
            encoding='cp949',
            skiprows=15,               # 메타데이터 줄 건너뛰기
            usecols=columns_to_use     # 필요한 컬럼만 불러오기
        )
        df_list.append(df)
        print(f"읽기 완료: {os.path.basename(file)}")
    except Exception as e:
        print(f"오류 발생: {os.path.basename(file)} — {e}")
# --- 여기까지 전국
# 아래부터 강남만
csv_files = glob.glob(os.path.join(gangnam_path, "*.csv"))
# 각 CSV 파일을 순회하며 읽기
for file in csv_files:
    try:
        df = pd.read_csv(
            file,
            encoding='cp949',
            skiprows=15,               # 메타데이터 줄 건너뛰기
            usecols=columns_to_use     # 필요한 컬럼만 불러오기
        )
        df_list.append(df)
        print(f"읽기 완료: {os.path.basename(file)}")
    except Exception as e:
        print(f"오류 발생: {os.path.basename(file)} — {e}")




# 데이터프레임 병합
df = pd.concat(df_list, ignore_index=True)

# 병합된 결과 출력
print(f"\n 병합 완료: 총 {len(df)}건")

읽기 완료: 아파트(매매)_실거래가_20250915192726.csv
읽기 완료: 아파트(매매)_실거래가_20250915192733.csv
읽기 완료: 아파트(매매)_실거래가_20250915192719.csv
읽기 완료: 아파트(매매)_실거래가_20250915192730.csv
읽기 완료: 아파트(매매)_실거래가_20250915192709.csv
읽기 완료: 아파트(매매)_실거래가_20250915192735.csv
읽기 완료: 아파트(매매)_실거래가_20250915192723.csv
읽기 완료: 아파트(매매)_실거래가_20250915192750.csv
읽기 완료: 아파트(매매)_실거래가_20250915192747.csv
읽기 완료: 아파트(매매)_실거래가_20250915192743.csv
읽기 완료: 아파트(매매)_실거래가_20250915192759.csv
읽기 완료: 아파트(매매)_실거래가_20250915192808.csv
읽기 완료: 아파트(매매)_실거래가_20250915192738.csv
읽기 완료: 아파트(매매)_실거래가_20250915192714.csv
읽기 완료: 아파트(매매)_실거래가_20250915192703.csv

 병합 완료: 총 64057건


In [46]:
min_contract = df['계약년월'].min()
max_contract = df['계약년월'].max()
print(f"데이터는 {min_contract} ~ {max_contract}까지의 거래로 이루어져 있습니다.")

데이터는 201007 ~ 202506까지의 거래로 이루어져 있습니다.


# 면적당 단가 계산

In [47]:
df['거래금액(만원)'] = df['거래금액(만원)'].str.replace(',', '').astype(int)
df['면적당 단가(만원)'] = df['거래금액(만원)'] / df['전용면적(㎡)']

In [48]:
df.head()

,시군구,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),동,층,건축년도,도로명,면적당 단가(만원)
0,서울특별시 강남구 대치동,래미안대치팰리스,84.990,201606,30,140000,-,3,2015,삼성로51길 37,1647.252618
1,서울특별시 강남구 대치동,대치아이파크,59.960,201606,30,97000,-,13,2008,선릉로 222,1617.745163
2,서울특별시 강남구 삼성동,삼성동힐스테이트 1단지,31.402,201606,30,64500,-,12,2008,학동로68길 29,2054.009299
3,서울특별시 강남구 수서동,까치마을,49.500,201606,30,62500,-,4,1993,광평로19길 10,1262.626263
4,서울특별시 강남구 논현동,두산위브1단지,84.993,201606,30,81000,-,5,2004,학동로46길 38,953.019660


# 아파트 나이 계산

In [49]:
df['계약년도'] = df['계약년월'].astype(str).str[:4].astype(int)
df['아파트 나이'] = df['계약년도'] - df['건축년도']

# 거래 순으로 나열 및 필요 없는 컬럼 삭졔

In [50]:
# 계약연-월-일을 기준으로 시계열 정렬
df['계약일자'] = df['계약년월'].astype(str) + df['계약일'].astype(str).str.zfill(2)
df['계약일자'] = pd.to_datetime(df['계약일자'], format='%Y%m%d')

df = df.sort_values('계약일자').reset_index(drop=True)
df.drop(['시군구','계약년월','계약일','동','계약년도','거래금액(만원)'], axis=1, inplace=True)

In [51]:
df.head()

,단지명,전용면적(㎡),층,건축년도,도로명,면적당 단가(만원),아파트 나이,계약일자
0,한화진넥스빌,39.200,15,2001,언주로86길 11,459.183673,9,2010-07-01
1,도곡스타클래스,111.380,14,2007,남부순환로 2615,650.924762,3,2010-07-01
2,경남아너스빌,81.013,3,2002,언주로85길 13,678.903386,8,2010-07-01
3,까치마을,39.600,7,1993,광평로19길 10,833.333333,17,2010-07-01
4,우정에쉐르멤버스,36.190,4,2004,선릉로87길 14,552.666482,6,2010-07-01


In [52]:
from pathlib import Path

OUTPUT_PATH = Path('../../data/interim/apt/gang_nam_apt_with_long_lat.csv')  

# === 좌표 변환 === #
headers = {'Authorization': f'KakaoAK {api_key}'}
#Authorization: KakaoAK ${REST_API_KEY}"
def get_coords(address):
    res = requests.get(
        "https://dapi.kakao.com/v2/local/search/address.json",
        headers=headers,
        params={'query': address}
    )
    if res.status_code == 200 and res.json()['documents']:
        doc = res.json()['documents'][0]
        return doc['x'], doc['y']
    return None, None

longitudes, latitudes = [], []
for address in tqdm(df['도로명'], desc="좌표 변환 중"):
    x, y = get_coords(address)
    longitudes.append(x)
    latitudes.append(y)

df['경도'] = longitudes
df['위도'] = latitudes

# === 최종 정제 및 저장 === #

df['면적당 단가(만원)'] = np.log(df['면적당 단가(만원)'])

# === 수집 못한 위도 경도는 삭제 === #
df.dropna(inplace=True)

# 디렉토리 없으면 생성
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

print(f"✅ 저장 완료: {OUTPUT_PATH}")

좌표 변환 중: 100%|█████████████████████| 64057/64057 [1:19:10<00:00, 13.49it/s]


✅ 저장 완료: ../../data/interim/apt/gang_nam_apt_with_long_lat.csv


In [57]:
df['계약일자'].max()

Timestamp('2025-06-30 00:00:00')

In [59]:
df['계약일자'].min()

Timestamp('2010-07-01 00:00:00')

In [64]:
df = df[df['계약일자'] >= '2018-07-01']

In [65]:
df.to_csv(OUTPUT_PATH, index=False)

print(f"✅ 저장 완료: {OUTPUT_PATH}")

✅ 저장 완료: ../../data/interim/apt/gang_nam_apt_with_long_lat.csv
